# 05 · Phase 4: multimodal fusion
Combines the best visual (Phase 2) and audio (Phase 3) models, trained on FakeAVCeleb, and
compares everything on the **same test set** with the **overall label**: fake if the face OR the voice is fake.

| Model | What it is |
|---|---|
| visual_only | Phase 2 face model alone |
| audio_only | Phase 3 voice model alone |
| late | no training: best of weighted-average / max / noisy-OR, tuned on validation |
| intermediate | embeddings of both branches → MLP; branches frozen first, then fine-tuned; modality dropout 0.2 |

**Needs:** `checkpoints/visual_fakeavceleb/best.pt` (notebook 03 with `DATASET="fakeavceleb"`) and
`checkpoints/audio_fakeavceleb/best.pt` (notebook 04, step 2). T4 GPU runtime.

In [ ]:
# --- Session setup: run this first, every time you open the notebook ---
REPO_URL = "https://github.com/tanishq1101/robust-multimodal-deepfake.git"  # "" = use code copied to Drive

import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

if REPO_URL:
    CODE = "/content/DeepFake"
    if os.path.exists(CODE):
        subprocess.run(["git", "-C", CODE, "pull"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, CODE], check=True)
else:
    # No GitHub repo yet: upload the project folder to Google Drive at MyDrive/deepfake_project/code
    CODE = "/content/drive/MyDrive/deepfake_project/code"
    assert os.path.exists(CODE), f"Upload the project folder to {CODE} or set REPO_URL"

os.chdir(CODE)
!pip install -q -r requirements-colab.txt && pip install -q -e .
sys.path.insert(0, f"{CODE}/src")

from deepfake.config import load_config
cfg = load_config()
print("data_root (Drive):", cfg["data_root"])
print("raw data (VM disk):", cfg["paths"]["raw"])

In [ ]:
import json, pandas as pd
from pathlib import Path
from IPython.display import Image, Markdown, display
CKPT = Path(cfg["paths"]["checkpoints"])
VISUAL = CKPT / "visual_fakeavceleb" / "best.pt"
AUDIO = CKPT / "audio_fakeavceleb" / "best.pt"
for p in (VISUAL, AUDIO):
    print(("OK      " if p.exists() else "MISSING ") + str(p))
missing = [str(p) for p in (VISUAL, AUDIO) if not p.exists()]
if missing:
    raise FileNotFoundError("Train the PyTorch branches in notebooks 03 and 04 first: " + ", ".join(missing))

## Late fusion (calibration only, takes ~1 min)

In [ ]:
!python -m deepfake.training.train_fusion --fusion late --visual-ckpt {VISUAL} --audio-ckpt {AUDIO} --workers 2
display(pd.read_csv(CKPT / "fusion_late_fakeavceleb" / "calibration.csv").head(10).round(4))

## Intermediate fusion (training)

In [ ]:
!python -m deepfake.training.train_fusion --fusion intermediate --visual-ckpt {VISUAL} --audio-ckpt {AUDIO} --epochs 10 --batch-size 8 --workers 2
display(Image(str(CKPT / "fusion_intermediate_fakeavceleb" / "curves.png")))

## Comparison table: the core result of the project

In [ ]:
models = {
    "visual_only": VISUAL,
    "audio_only": AUDIO,
    "late": CKPT / "fusion_late_fakeavceleb" / "best.pt",
    "intermediate": CKPT / "fusion_intermediate_fakeavceleb" / "best.pt",
}
args = " ".join(f"--model {k}={v}" for k, v in models.items() if Path(v).exists())
!python -m deepfake.evaluation.compare --dataset fakeavceleb {args} --workers 2
OUT = Path(cfg["paths"]["outputs"]) / "compare_fakeavceleb_test"
display(Markdown((OUT / "comparison.md").read_text()))
display(Image(str(OUT / "comparison.png")))

**How to read it:** visual_only should fail on *RealVideo-FakeAudio* and audio_only on
*FakeVideo-RealAudio*. Fusion should handle all four categories. Copy `comparison.md` into
`reports/results.md`.